# Tutorial 7 — SMARTS Substructure Search
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

SMARTS (SMILES Arbitrary Target Specification) extends SMILES with query atoms and bonds. It is the standard for substructure searching, functional group detection, and reaction SMARTS in retrosynthesis.

In [ ]:
!pip install rdkit pandas -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import Image
import pandas as pd

# Define pharmacophore/functional group patterns
patterns = {
    "Carboxylic acid":     "[CX3](=O)[OX2H1]",
    "Ester":               "[CX3](=O)[OX2][CX4]",
    "Amide":               "[CX3](=O)[NX3H]",
    "Primary amine":       "[NX3;H2]",
    "Secondary amine":     "[NX3;H1;!$(NC=O)]",
    "Aromatic ring":       "c1ccccc1",
    "Hydroxyl":            "[OX2H]",
    "Sulfonamide":         "[NX3][SX4](=O)(=O)",
    "Halogen":             "[F,Cl,Br,I]",
    "Nitro":               "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "Thiophene":           "c1ccsc1",
    "Imidazole":           "c1cn[nH]c1",
}

library = {
    "Aspirin":       "CC(=O)Oc1ccccc1C(=O)O",
    "Paracetamol":   "CC(=O)Nc1ccc(O)cc1",
    "Morphine":      "OC1=CC=C2CC3N(CCC34CCc5c4cc(O)c(OC)c5)C2=C1",
    "Ciprofloxacin": "OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O",
    "Metformin":     "CN(C)C(=N)NC(=N)N",
    "Furosemide":    "NS(=O)(=O)c1cc(C(=O)O)c(NCc2ccco2)cc1Cl",
    "Omeprazole":    "CC1=CN=C(CS(=O)c2nc3cc(OC)ccc3[nH]2)C(OC)=C1",
}

rows = []
for mol_name, smi in library.items():
    mol = Chem.MolFromSmiles(smi)
    row = {"Molecule": mol_name}
    for pat_name, smarts in patterns.items():
        pat = Chem.MolFromSmarts(smarts)
        matches = mol.GetSubstructMatches(pat)
        row[pat_name] = len(matches) if matches else 0
    rows.append(row)

df = pd.DataFrame(rows).set_index("Molecule")
print(df.to_string())

## Highlight matching atoms

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image
import io

def highlight_match(mol, smarts, size=(400, 300)):
    pat    = Chem.MolFromSmarts(smarts)
    matches= mol.GetSubstructMatches(pat)
    hit_atoms = set()
    hit_bonds = set()
    for match in matches:
        hit_atoms.update(match)
        for b in mol.GetBonds():
            if b.GetBeginAtomIdx() in match and b.GetEndAtomIdx() in match:
                hit_bonds.add(b.GetIdx())
    drawer = rdMolDraw2D.MolDraw2DCairo(*size)
    drawer.drawOptions().addAtomIndices = False
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, mol,
        highlightAtoms=list(hit_atoms),
        highlightBonds=list(hit_bonds),
    )
    drawer.FinishDrawing()
    return Image.open(io.BytesIO(drawer.GetDrawingText()))

# Show sulfonamide match in Furosemide
mol_furo = Chem.MolFromSmiles(library["Furosemide"])
img = highlight_match(mol_furo, "[NX3][SX4](=O)(=O)")
img.save("sulfonamide_highlight.png")
display(img)
print("Highlighted sulfonamide in Furosemide")

## SMARTS for reaction matching

In [ ]:
from rdkit.Chem import AllChem

# Ester hydrolysis reaction SMARTS
rxn = AllChem.ReactionFromSmarts("[CX3:1](=O)[OX2:2][CX4:3]>>[CX3:1](=O)[OX2H:2].[CX4:3]")
print("Reaction SMARTS defined: Ester hydrolysis")

aspirin_mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
products = rxn.RunReactants((aspirin_mol,))
if products:
    for i, prod_set in enumerate(products):
        for j, prod in enumerate(prod_set):
            try:
                Chem.SanitizeMol(prod)
                print(f"  Product {i},{j}: {Chem.MolToSmiles(prod)}")
            except:
                pass

## Key takeaways
- SMARTS wildcards: `[CX3]` = sp2 carbon, `[NX3;H2]` = primary amine, `[$(...)]` = recursive SMARTS
- Atom map numbers (`:1`, `:2`) in reaction SMARTS track which atoms go where
- SMARTS is indispensable for: toxicophore flagging, scaffold hopping, retrosynthesis
- RDKit's `HasSubstructMatch` returns bool; `GetSubstructMatches` returns all match tuples